## Импорты

In [1]:
try:
    import google.colab
    %pip install pandas torch transformers datasets evaluate sacrebleu bert-score sentence-transformers rapidfuzz deep-translator tqdm numpy matplotlib PyYAML pymorphy3# translatepy
    %pip install --upgrade tiktoken protobuf sentencepiece
    %pip install accelerate>=1.1.0
    is_colab = True
except:
    is_colab = False

if is_colab:
    !git clone https://github.com/AbonentVneSeti/text_processing_final_project
    %cd text_processing_final_project

In [2]:
import os
import sys
sys.path.append('..')
from utils import load_config
import pandas as pd
import importlib

config = load_config("config.yaml")
print("Конфиг загружен")

Конфиг загружен


## Создание датасета

### Backtranslation

In [ ]:
if not os.path.exists("data/paraphrases_raw.parquet"):
    from dataset.create_dataset import tapaco, backtranslation

    #tapaco
    tapaco_params = config["source_params"]["tapaco"]
    df_tapaco = tapaco.load_or_create(tapaco_params)
    
    unique_originals = df_tapaco[['original']].drop_duplicates()['original'].tolist()
    unique_originals = unique_originals

    #backtranslation
    bt_params = config["source_params"]["backtranslation"].copy()
    bt_params['sentences'] = unique_originals
    df_bt = backtranslation.load_or_create(bt_params)

    #save
    output_path = "data/paraphrases_raw.parquet"
    df_bt.to_parquet(output_path, index=False)
    print(f"Датасет сохранён в {output_path}")

d:\programming projects\3 course\text_processing_final_project\.conda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Найдено 10000 уже обработанных предложений, продолжаем...


Back-translating:   0%|          | 3/2413 [01:36<21:24:42, 31.98s/it]

## Предобработка

In [ ]:
df_bt = pd.read_parquet("data/paraphrases_raw.parquet")
print(f"Загружено {len(df_bt)} пар")

In [ ]:
from dataset.prepare_dataset.prepare import prepare_dataset

df_clean = prepare_dataset(df_bt, config["preprocessing"])
print(f"После очистки: {len(df_clean)} пар")

## Сохранение

In [ ]:
output_path = "data/paraphrases_clean.parquet"
df_clean.to_parquet(output_path, index=False)
print(f"Датасет сохранён в {output_path}")

In [ ]:
if is_colab:
    from google.colab import drive
    drive.mount('/content/drive')
    !cp {output_path} /content/drive/MyDrive/
    print("Скопировано на Google Drive")